# Can Saliency Maps Flag Chest Radiograph Classifier Errors at a New Hospital? An External Validation Study
## XAI Analysis
## Aashish Chapagai (Last Edited: October 2026)

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json
import cv2
import shap
import numpy as np
import pandas as pd
import tensorflow as tf
import keras_hub
import matplotlib.pyplot as plt
from lime import lime_image
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

tf.get_logger().setLevel('ERROR')

def no_progress(iterable, *args, **kwargs):
    return iterable
lime_image.tqdm = no_progress

os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f"Running on {gpus[0].name}.")

In [ ]:
MODEL = 'densenet'
MODEL_NAME, MODEL_PATH, PREFIX = {
    'densenet': ('DenseNet-121', 'models/cnn_base.keras', ''),
    'vit': ('ViT-B/16', 'models/vit_base.keras', 'vit_'),
}[MODEL]
SEED = 42
N_BOOTSTRAP = 1000
BATCH_SIZE = 8 if MODEL == 'vit' else 16
MASK_QUALITY_MIN = 0.7
MIN_POSITIVES = 10
MIN_EVENTS = 20
RANDOMISATION_IMAGES = 200
CROSS_METHOD_IMAGES = 500
LIME_SAMPLES = 1000
SHAP_BACKGROUND = 50
SHAP_SAMPLES = 100
HARMONISED = {
    'Atelectasis': ['Atelectasis'],
    'Cardiomegaly': ['Cardiomegaly'],
    'Consolidation': ['Consolidation'],
    'Edema': ['Edema'],
    'Effusion': ['Effusion'],
    'Emphysema': ['Emphysema'],
    'Fibrosis': ['Fibrosis'],
    'Infiltration': ['Infiltration'],
    'Lesion': ['Mass', 'Nodule'],
    'Pleural thickening': ['Pleural_Thickening'],
    'Pneumonia': ['Pneumonia'],
    'Pneumothorax': ['Pneumothorax'],
}
SCORES = ['Confidence margin', 'TTA agreement', 'Anatomical enrichment', 'Margin + TTA', 'Margin + TTA + anatomy']
COMBINERS = (('Margin + TTA', ['abs_margin', 'tta_gap']), ('Margin + TTA + anatomy', ['abs_margin', 'tta_gap', 'region_enrichment']))
COLOURS = {'NIH': '#4C72B0', 'CheXpert': '#DD8452', 'VinDr-CXR': '#55A868'}
LIME_SETTINGS = {'top_labels': None, 'hide_color': 0, 'num_samples': LIME_SAMPLES, 'batch_size': 64, 'random_seed': SEED}
tf.keras.utils.set_random_seed(SEED)

In [ ]:
if MODEL == 'vit':
    tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')
classifier = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f"Loaded {MODEL_NAME} from {MODEL_PATH}.")
disease_classes = np.load('models/disease_classes.npy').tolist()
class_index = {name: i for i, name in enumerate(disease_classes)}
thresholds = np.load(f'models/{PREFIX}thresholds_macro.npy')
Y_test = np.load('models/Y_test.npy')
nih_columns = [[class_index[s] for s in sources] for sources in HARMONISED.values()]
nih_labels = np.stack([Y_test[:, cols].max(axis=1) for cols in nih_columns], axis=1).astype(np.uint8)
datasets = {'NIH': {
    'images': np.load('models/X_test.npy', mmap_mode='r'),
    'labels': nih_labels,
    'findings': list(HARMONISED),
    'patients': np.load('models/test_patient_ids.npy', allow_pickle=True).astype(str),
    'anatomy': np.load('models/anatomy_nih.npy', mmap_mode='r'),
    'quality': np.load('models/anatomy_nih_quality.npy'),
}}
for name, stem in (('CheXpert', 'chexpert'), ('VinDr-CXR', 'vindr')):
    if not os.path.exists(f'models/X_{stem}.npy'):
        print(f"{name} arrays not found. Skipping it.")
        continue
    metadata = pd.read_csv(f'models/metadata_{stem}.csv')
    has_anatomy = os.path.exists(f'models/anatomy_{stem}.npy')
    anatomy = np.load(f'models/anatomy_{stem}.npy', mmap_mode='r') if has_anatomy else None
    quality = np.load(f'models/anatomy_{stem}_quality.npy') if has_anatomy else np.full(len(metadata), np.nan, dtype=np.float32)
    datasets[name] = {
        'images': np.load(f'models/X_{stem}.npy', mmap_mode='r'),
        'labels': np.load(f'models/Y_{stem}.npy'),
        'findings': np.load(f'models/{stem}_findings.npy').tolist(),
        'patients': metadata['patient'].astype(str).to_numpy(),
        'anatomy': anatomy,
        'quality': quality,
    }
IMG_SIZE = datasets['NIH']['images'].shape[1]
for name, data in datasets.items():
    usable = int((data['quality'] >= MASK_QUALITY_MIN).sum())
    print(f"{name}: {len(data['labels'])} images from {len(np.unique(data['patients']))} patients.")
    print(f"{name}: {len(data['findings'])} harmonised findings, usable anatomy for {usable}.")

In [ ]:
def densenet_cams(model):
    feature_model = tf.keras.Model(model.input, [model.get_layer('gap').input, model.output])
    class_kernel = model.get_layer('logits').kernel.numpy().astype(np.float32)

    def forward(images):
        tensor = tf.convert_to_tensor(images, dtype=tf.float32)
        feature_maps, probabilities = feature_model(tensor, training=False)
        cams = np.einsum('bhwk,kc->bhwc', feature_maps.numpy().astype(np.float32), class_kernel)
        return np.maximum(cams, 0.0), probabilities.numpy()
    return forward

def vit_cams(model):
    backbone = model.get_layer('vit_b16_base')
    encoder = backbone.get_layer('vit_encoder')
    embed = backbone.get_layer('vit_patching_and_embedding')
    prepare = tf.keras.Model(model.input, model.get_layer('to_rgb').output)
    dropout, head = model.get_layer('head_dropout'), model.get_layer('logits')
    *blocks, last = encoder.encoder_layers
    side = backbone.image_shape[0] // backbone.patch_size

    def forward(images):
        tokens = embed(prepare(tf.convert_to_tensor(images, dtype=tf.float32)))
        for block in blocks:
            tokens = block(tokens)
        with tf.GradientTape(persistent=True) as tape:
            normed = last.layer_norm_1(tokens)
            tape.watch(normed)
            attended = tokens + last.mha(normed, normed)
            encoded = encoder.layer_norm(attended + last.mlp(last.layer_norm_2(attended)))
            logits = head(dropout(encoded[:, 0]))
            class_logits = [logits[:, c] for c in range(logits.shape[-1])]
        patches = tf.cast(normed[:, 1:], tf.float32)
        cams = []
        for score in class_logits:
            weights = tf.reduce_sum(tf.cast(tape.gradient(score, normed)[:, 1:], tf.float32), axis=1)
            cams.append(tf.einsum('bpk,bk->bp', patches, weights))
        cams = tf.reshape(tf.stack(cams, axis=-1), (-1, side, side, len(cams)))
        return np.maximum(cams.numpy(), 0.0), tf.sigmoid(logits).numpy()
    return forward

make_forward = vit_cams if MODEL == 'vit' else densenet_cams
forward = make_forward(classifier)

def run_model(images, batch_size=BATCH_SIZE):
    n = len(images)
    upright = np.zeros((n, len(disease_classes)), dtype=np.float32)
    flipped = np.zeros_like(upright)
    cams = None
    for start in range(0, n, batch_size):
        batch = np.asarray(images[start:start + batch_size], dtype=np.float32)
        stop = start + len(batch)
        cam_upright, upright[start:stop] = forward(batch)
        mirrored = np.ascontiguousarray(batch[:, :, ::-1])
        cam_flipped, flipped[start:stop] = forward(mirrored)
        if cams is None:
            cams = np.zeros((n,) + cam_upright.shape[1:], dtype=np.float16)
        cams[start:stop] = 0.5 * (cam_upright + cam_flipped[:, :, ::-1])
        if (start // batch_size) % 200 == 0:
            print(f"  Explained {start} of {n} images", flush=True)
    return {'upright': upright, 'flipped': flipped, 'cams': cams}

In [ ]:
check = np.asarray(datasets['NIH']['images'][:8], dtype=np.float32)
cams, probabilities = forward(check)
print(f"Grad-CAM maps are {cams.shape[1]}x{cams.shape[2]} before upsampling.")
if MODEL == 'vit':
    gap = np.abs(probabilities - classifier(check, training=False).numpy()).max()
    print(f"Largest gap between the Grad-CAM forward pass and the model's own predictions: {gap:.2e}")
else:
    feature_model = tf.keras.Model(classifier.input, classifier.get_layer('gap').input)
    logits_layer = classifier.get_layer('logits')
    feature_maps = tf.cast(feature_model(check, training=False), tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(feature_maps)
        logits = logits_layer(tf.reduce_mean(feature_maps, axis=[1, 2]))
        class_logits = [logits[:, c] for c in range(len(disease_classes))]
    gradcams = []
    for c in range(len(disease_classes)):
        gradients = tape.gradient(class_logits[c], feature_maps)
        weights = tf.reduce_mean(gradients, axis=[1, 2])
        gradcams.append(tf.nn.relu(tf.einsum('bhwk,bk->bhw', feature_maps, weights)).numpy())
    gradcams = np.stack(gradcams, axis=-1) * feature_maps.shape[1] * feature_maps.shape[2]
    gap = np.abs(gradcams - cams).max()
    print(f"Largest gap between gradient-based Grad-CAM and the closed form: {gap:.2e}")
print(f"Peak Grad-CAM value: {cams.max():.2f}")

In [ ]:
outputs = {}
for name, data in datasets.items():
    print(f"Explaining {name}.")
    outputs[name] = run_model(data['images'])
    np.savez(f"models/{PREFIX}xai_outputs_{name.lower().replace('-', '')}.npz", **outputs[name])
reference = np.load(f'models/{PREFIX}test_probabilities.npy')
nih_probabilities = 0.5 * (outputs['NIH']['upright'] + outputs['NIH']['flipped'])
probability_gap = np.abs(nih_probabilities - reference).max()
print(f"Largest gap to the training notebook's test probabilities: {probability_gap:.4f}")

In [ ]:
def anatomy_attribution(cams, anatomy, quality):
    n, n_classes = len(cams), cams.shape[-1]
    parts = ('region', 'lung')
    shares = {part: np.full((n, n_classes), np.nan, dtype=np.float32) for part in parts}
    areas = {part: np.full(n, np.nan, dtype=np.float32) for part in parts}
    if anatomy is None:
        return shares, areas
    for i in np.flatnonzero(quality >= MASK_QUALITY_MIN):
        layers = np.asarray(anatomy[i])
        upsampled = cv2.resize(cams[i].astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        maps = upsampled.reshape(IMG_SIZE, IMG_SIZE, n_classes)
        total = maps.sum(axis=(0, 1))
        for part, mask in (('region', layers & 1), ('lung', (layers >> 1) & 1)):
            mask = mask.astype(np.float32)
            if mask.sum() == 0:
                continue
            inside = np.tensordot(mask, maps, axes=([0, 1], [0, 1]))
            share = inside / np.maximum(total, 1e-12)
            shares[part][i] = np.where(total > 0, share, np.nan)
            areas[part][i] = mask.mean()
    return shares, areas

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

In [ ]:
def decision_table(name, data, output):
    shares, areas = anatomy_attribution(output['cams'], data['anatomy'], data['quality'])
    probabilities = 0.5 * (output['upright'] + output['flipped'])
    margins = logit(probabilities) - logit(thresholds)[None, :]
    gaps = np.abs(output['upright'] - output['flipped'])
    rows = np.arange(len(probabilities))
    frames = []
    for f, finding in enumerate(data['findings']):
        sources = np.array([class_index[s] for s in HARMONISED[finding]])
        driver = sources[np.argmax(margins[:, sources], axis=1)]
        frames.append(pd.DataFrame({
            'dataset': name,
            'row': rows,
            'patient': data['patients'],
            'finding': finding,
            'driver': driver,
            'label': data['labels'][:, f].astype(int),
            'probability': probabilities[rows, driver],
            'margin': margins[rows, driver],
            'tta_gap': gaps[rows, driver],
            'region_share': shares['region'][rows, driver],
            'lung_share': shares['lung'][rows, driver],
            'region_area': areas['region'],
            'lung_area': areas['lung'],
        }))
    table = pd.concat(frames, ignore_index=True)
    table['decision'] = (table['margin'] > 0).astype(int)
    table['correct'] = (table['decision'] == table['label']).astype(int)
    for part in ('region', 'lung'):
        area = table[f'{part}_area']
        enrichment = table[f'{part}_share'] / area.where(area > 0)
        undefined = area.gt(0) & table[f'{part}_share'].isna()
        table[f'{part}_enrichment'] = enrichment.mask(undefined, 1.0)
    return table

def matrix(table, column, findings):
    return table[column].to_numpy().reshape(len(findings), -1).T

In [ ]:
tables = {name: decision_table(name, datasets[name], outputs[name]) for name in datasets}
decisions = pd.concat(tables.values(), ignore_index=True)
decisions.to_csv(f'models/{PREFIX}xai_decisions.csv', index=False)
for name, table in tables.items():
    flagged = table[table['decision'] == 1]
    true_flags = int(flagged['correct'].sum())
    print(f"{name}: {len(flagged)} flagged findings, {true_flags} true and {len(flagged) - true_flags} false.")
    print(f"{name}: anatomical enrichment available for {int(flagged['region_enrichment'].notna().sum())}.")

In [ ]:
def patient_groups(patients):
    _, inverse = np.unique(np.asarray(patients).astype(str), return_inverse=True)
    order = np.argsort(inverse, kind='stable')
    return np.split(order, np.cumsum(np.bincount(inverse))[:-1])

def bootstrap(statistics, groups, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    samples = {key: [] for key in statistics}
    for _ in range(n_bootstrap):
        rows = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        for key, (statistic, arguments) in statistics.items():
            value = statistic(rows, **arguments)
            if value is not None and np.isfinite(value):
                samples[key].append(value)
    intervals = {}
    for key, values in samples.items():
        enough = len(values) >= n_bootstrap // 2
        intervals[key] = np.percentile(values, [2.5, 97.5]) if enough else np.array([np.nan, np.nan])
    return intervals

def safe_auc(truth, score):
    return roc_auc_score(truth, score) if 0 < truth.sum() < len(truth) else None

def safe_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else None

def auc_against(rows, correct, values):
    return safe_auc(correct[rows], values[rows])

def auc_at(rows, labels, margins, column):
    return safe_auc(labels[rows, column], margins[rows, column])

def sensitivity_at(rows, labels, margins, column):
    return safe_mean(margins[rows, column][labels[rows, column] == 1] > 0)

def specificity_at(rows, labels, margins, column):
    return safe_mean(margins[rows, column][labels[rows, column] == 0] <= 0)

def macro_auc_at(rows, labels, margins, columns):
    return safe_mean([safe_auc(labels[rows, c], margins[rows, c]) or np.nan for c in columns])    

In [ ]:
prepared = {}
for name, data in datasets.items():
    labels = matrix(tables[name], 'label', data['findings'])
    margins = matrix(tables[name], 'margin', data['findings'])
    usable = [f for f in range(len(data['findings'])) if labels[:, f].sum() >= MIN_POSITIVES]
    prepared[name] = (labels, margins, usable)

In [ ]:
transfer = []
for name, data in datasets.items():
    findings = data['findings']
    labels, margins, usable = prepared[name]
    statistics = {}
    macro_columns = {'Macro average': usable}
    for f in usable:
        bound = {'labels': labels, 'margins': margins, 'column': f}
        statistics[(findings[f], 'AUROC')] = (auc_at, bound)
        statistics[(findings[f], 'Sensitivity')] = (sensitivity_at, bound)
        statistics[(findings[f], 'Specificity')] = (specificity_at, bound)
    if name == 'NIH':
        for other in datasets:
            other_findings = {datasets[other]['findings'][f] for f in prepared[other][2]}
            shared = [f for f in usable if findings[f] in other_findings]
            if other != 'NIH' and 0 < len(shared) < len(usable):
                macro_columns[f'Macro average, shared with {other}'] = shared
    for label, columns in macro_columns.items():
        statistics[(label, 'AUROC')] = (macro_auc_at, {'labels': labels, 'margins': margins, 'columns': columns})
    ci = bootstrap(statistics, patient_groups(data['patients']))
    everything = np.arange(len(labels))
    for key, (statistic, arguments) in statistics.items():
        columns = [findings.index(key[0])] if key[0] in findings else macro_columns[key[0]]
        transfer.append({
            'dataset': name,
            'finding': key[0],
            'metric': key[1],
            'value': statistic(everything, **arguments),            
            'low': ci[key][0],
            'high': ci[key][1],
            'positives': int(labels[:, columns].sum()),
        })
transfer = pd.DataFrame(transfer)
transfer.to_csv(f'models/{PREFIX}xai_transfer.csv', index=False)
for name in datasets:
    print(f"\n{name} (95% patient-level bootstrap CI)")
    part = transfer[transfer['dataset'] == name]
    for finding in pd.unique(part['finding']):
        rows = part[part['finding'] == finding].set_index('metric')
        print(f"  {finding}")
        for metric in rows.index:
            value, low, high = rows.at[metric, 'value'], rows.at[metric, 'low'], rows.at[metric, 'high']
            print(f"    {metric} {value:.3f} [{low:.3f}, {high:.3f}]")

In [ ]:
reported = set(transfer['finding'])
shared_rows = sorted(f for f in reported if f.startswith('Macro average, shared'))
order = [f for f in HARMONISED if f in reported] + ['Macro average'] + shared_rows
fig, ax = plt.subplots(figsize=(8, 0.45 * len(order) + 1.5))
for offset, name in zip(np.linspace(-0.2, 0.2, len(datasets)), datasets):
    part = transfer[(transfer['dataset'] == name) & (transfer['metric'] == 'AUROC')].set_index('finding')
    present = [f for f in order if f in part.index]
    values = part.loc[present, 'value'].to_numpy(dtype=float)
    low = part.loc[present, 'low'].to_numpy(dtype=float)
    high = part.loc[present, 'high'].to_numpy(dtype=float)
    errors = np.nan_to_num(np.vstack([values - low, high - values]))
    positions = [order.index(f) + offset for f in present]
    ax.errorbar(values, positions, xerr=errors, fmt='o', color=COLOURS[name], label=name, capsize=2, markersize=4)
ax.axvline(0.5, color='grey', linestyle=':', linewidth=1)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.invert_yaxis()
ax.set_xlabel('ROC-AUC with 95% patient-level bootstrap CI')
ax.set_title(f'{MODEL_NAME} classification transfer from NIH to external sites')
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(f'figures/{PREFIX}classification_transfer.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def upsample(cam):
    return cv2.resize(cam.astype(np.float32), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def box_mask(x0, y0, x1, y1):
    mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    top, bottom = int(np.floor(max(y0, 0))), int(np.ceil(min(y1, IMG_SIZE)))
    left, right = int(np.floor(max(x0, 0))), int(np.ceil(min(x1, IMG_SIZE)))
    mask[top:bottom, left:right] = True
    return mask

def overlap(prediction, truth):
    union = np.logical_or(prediction, truth).sum()
    return float(np.logical_and(prediction, truth).sum() / union) if union else 0.0

def localise(saliency, truth):
    if saliency.max() <= 0:
        return 0.0, 0.0
    peak = np.unravel_index(np.argmax(saliency), saliency.shape)
    scaled = np.uint8(np.round(255 * saliency / saliency.max()))
    _, binary = cv2.threshold(scaled, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return float(truth[peak]), overlap(binary.astype(bool), truth)

In [ ]:
ground_truth = {name: {} for name in datasets}

def add_truth(store, name, row, finding, mask):
    key = (int(row), str(finding))
    store[name][key] = store[name].get(key, np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)) | mask

for name, path in (('NIH', 'models/gt_boxes_nih.csv'), ('VinDr-CXR', 'models/gt_boxes_vindr.csv')):
    if name in datasets and os.path.exists(path):
        for box in pd.read_csv(path).itertuples():
            add_truth(ground_truth, name, box.row, box.finding, box_mask(box.x0, box.y0, box.x1, box.y1))
human = {'CheXpert': {}}
if 'CheXpert' in datasets:
    for path, store in (('models/gt_masks_chexpert.npz', ground_truth), ('models/hb_masks_chexpert.npz', human)):
        stored = np.load(path)
        for row, finding, mask in zip(stored['rows'], stored['findings'], stored['masks']):
            add_truth(store, 'CheXpert', row, finding, mask)
    points = {(int(p.row), p.finding): (p.x, p.y) for p in pd.read_csv('models/hb_points_chexpert.csv').itertuples()}

In [ ]:
localisation = []
for name, truths in ground_truth.items():
    findings = datasets[name]['findings']
    drivers = matrix(tables[name], 'driver', findings)
    decided = matrix(tables[name], 'decision', findings)
    for (row, finding), truth in truths.items():
        if finding not in findings:
            continue
        f = findings.index(finding)
        saliency = upsample(outputs[name]['cams'][row, :, :, drivers[row, f]])
        hit, iou = localise(saliency, truth)
        localisation.append({
            'dataset': name,
            'method': 'Grad-CAM',
            'row': row,
            'patient': datasets[name]['patients'][row],
            'finding': finding,
            'hit': hit,
            'iou': iou,
            'flagged': int(decided[row, f]),
            'benchmarked': int((row, finding) in human.get(name, {})),
        })
for key, truth in ground_truth.get('CheXpert', {}).items():
    if key in human['CheXpert']:
        point = points.get(key)
        if point is None:
            hit = np.nan
        else:
            y = int(np.clip(point[1], 0, IMG_SIZE - 1))
            x = int(np.clip(point[0], 0, IMG_SIZE - 1))
            hit = float(truth[y, x])
        localisation.append({
            'dataset': 'CheXpert',
            'method': 'Human benchmark',
            'row': key[0],
            'patient': datasets['CheXpert']['patients'][key[0]],
            'finding': key[1],
            'hit': hit,
            'iou': overlap(human['CheXpert'][key], truth),
            'flagged': 1,
            'benchmarked': 1,
        })
localisation = pd.DataFrame(localisation)
localisation.to_csv(f'models/{PREFIX}xai_localisation.csv', index=False)

In [ ]:
def mean_at(rows, values, selector=None):
    return safe_mean(values[rows] if selector is None else values[rows][selector[rows]])

def summarise_localisation(frame, label):
    hit = frame['hit'].to_numpy(dtype=float)
    iou = frame['iou'].to_numpy(dtype=float)
    finding = frame['finding'].to_numpy()
    statistics = {}
    statistics[('All findings', 'Hit rate')] = (mean_at, {'values': hit})
    statistics[('All findings', 'mIoU')] = (mean_at, {'values': iou})
    for name in pd.unique(finding):
        selector = finding == name
        if selector.sum() >= 5:
            statistics[(name, 'Hit rate')] = (mean_at, {'values': hit, 'selector': selector})
            statistics[(name, 'mIoU')] = (mean_at, {'values': iou, 'selector': selector})
    ci = bootstrap(statistics, patient_groups(frame['patient']))
    everything = np.arange(len(frame))
    summary = []
    for key, (statistic, arguments) in statistics.items():
        n = len(frame) if key[0] == 'All findings' else int((finding == key[0]).sum())
        summary.append({
            'group': label,
            'finding': key[0],
            'metric': key[1],
            'value': statistic(everything, **arguments),
            'low': ci[key][0],
            'high': ci[key][1],
            'n': n,
        })
    return summary

In [ ]:
localisation_groups = []
for name in datasets:
    gradcam_rows = localisation[localisation['method'] == 'Grad-CAM']
    model_rows = gradcam_rows[gradcam_rows['dataset'] == name]
    if len(model_rows):
        localisation_groups.append((f"{name} Grad-CAM", model_rows))
        localisation_groups.append((f"{name} Grad-CAM, flagged only", model_rows[model_rows['flagged'] == 1]))
if 'CheXpert' in datasets:
    benchmarked = localisation[(localisation['method'] == 'Grad-CAM') & (localisation['benchmarked'] == 1)]
    localisation_groups.append(('CheXpert Grad-CAM, benchmarked images', benchmarked))
    localisation_groups.append(('CheXpert human benchmark', localisation[localisation['method'] == 'Human benchmark']))
summary_rows = []
for label, frame in localisation_groups:
    if len(frame) >= 5:
        summary_rows.extend(summarise_localisation(frame, label))
localisation_summary = pd.DataFrame(summary_rows)
localisation_summary.to_csv(f'models/{PREFIX}xai_localisation_summary.csv', index=False)
for label in pd.unique(localisation_summary['group']):
    group_rows = localisation_summary[localisation_summary['group'] == label]
    part = group_rows[group_rows['finding'] == 'All findings'].set_index('metric')
    hit, iou = part.loc['Hit rate'], part.loc['mIoU']
    print(f"{label}: n={int(hit['n'])}")
    print(f"  Hit rate {hit['value']:.3f} [{hit['low']:.3f}, {hit['high']:.3f}]")
    print(f"  mIoU {iou['value']:.3f} [{iou['low']:.3f}, {iou['high']:.3f}]")

In [ ]:
plotted = [label for label in pd.unique(localisation_summary['group']) if 'flagged' not in label]
summarised = set(localisation_summary['finding'])
findings_plotted = [f for f in HARMONISED if f in summarised] + ['All findings']
fig, ax = plt.subplots(figsize=(11, 4.5))
width = 0.8 / max(len(plotted), 1)
for i, label in enumerate(plotted):
    group_rows = localisation_summary[localisation_summary['group'] == label]
    part = group_rows[group_rows['metric'] == 'Hit rate'].set_index('finding')
    present = [f for f in findings_plotted if f in part.index]
    values = part.loc[present, 'value'].to_numpy(dtype=float)
    low = part.loc[present, 'low'].to_numpy(dtype=float)
    high = part.loc[present, 'high'].to_numpy(dtype=float)
    errors = np.nan_to_num(np.vstack([values - low, high - values])) 
    offset = (i - (len(plotted) - 1) / 2) * width
    positions = [findings_plotted.index(f) + offset for f in present]
    ax.bar(positions, values, width, yerr=errors, capsize=2, label=label)
ax.set_xticks(range(len(findings_plotted)))
ax.set_xticklabels(findings_plotted, rotation=30, ha='right')
ax.set_ylabel('Pointing-game hit rate (95% CI)')
ax.set_ylim(0, 1)
ax.set_title(f'Does the {MODEL_NAME} Grad-CAM peak fall inside the radiologist annotation?')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'figures/{PREFIX}localisation_hit_rate.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
has_enrichment = decisions['region_enrichment'].notna()
flags = decisions[(decisions['decision'] == 1) & has_enrichment].copy()
flags['outcome'] = np.where(flags['correct'] == 1, 'True positive', 'False positive')
print("Anatomical enrichment of flagged findings:")
for name in datasets:
    part = flags[flags['dataset'] == name]
    if len(part) == 0:
        continue
    medians = part.groupby('outcome')['region_enrichment'].median()
    print(f"  {name}: n={len(part)}")
    print(f"  Median true positive {medians.get('True positive', np.nan):.3f}, median false positive {medians.get('False positive', np.nan):.3f}")
    print(f"  Share below chance {np.mean(part['region_enrichment'] < 1):.3f}")

In [ ]:
def two_sample_ci(a, a_groups, b, b_groups, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_bootstrap):
        rows_a = np.concatenate([a_groups[g] for g in rng.integers(0, len(a_groups), len(a_groups))])
        rows_b = np.concatenate([b_groups[g] for g in rng.integers(0, len(b_groups), len(b_groups))])
        values.append(np.median(b[rows_b]) - np.median(a[rows_a]))
    return np.percentile(values, [2.5, 97.5])

In [ ]:
shift_rows = []
nih_flags = flags[flags['dataset'] == 'NIH']
for name in [n for n in datasets if n != 'NIH']:
    external = flags[flags['dataset'] == name]
    if len(external) < MIN_EVENTS:
        continue
    for finding in ['All findings'] + [f for f in HARMONISED if f in set(external['finding'])]:
        all_findings = finding == 'All findings'
        a = nih_flags if all_findings else nih_flags[nih_flags['finding'] == finding]
        b = external if all_findings else external[external['finding'] == finding]
        if len(a) < MIN_EVENTS or len(b) < MIN_EVENTS:
            continue
        nih_values, nih_groups = a['region_enrichment'].to_numpy(), patient_groups(a['patient'])
        site_values, site_groups = b['region_enrichment'].to_numpy(), patient_groups(b['patient'])
        low, high = two_sample_ci(nih_values, nih_groups, site_values, site_groups)        
        nih_median = float(a['region_enrichment'].median())
        external_median = float(b['region_enrichment'].median())
        shift_rows.append({
            'external': name,
            'finding': finding,
            'nih_median': nih_median,
            'external_median': external_median,
            'difference': external_median - nih_median,
            'low': low,
            'high': high,
            'p_value': float(mannwhitneyu(a['region_enrichment'], b['region_enrichment']).pvalue),
            'n_nih': len(a),
            'n_external': len(b),
        })
shift = pd.DataFrame(shift_rows)
shift.to_csv(f'models/{PREFIX}xai_enrichment_shift.csv', index=False)
for row in shift.itertuples():
    print(f"  {row.external} vs NIH, {row.finding}: median {row.external_median:.3f} vs {row.nih_median:.3f}")
    print(f"  Difference {row.difference:+.3f} [{row.low:+.3f}, {row.high:+.3f}], Mann-Whitney p={row.p_value:.2g}")

In [ ]:
names_with_flags = [n for n in datasets if (flags['dataset'] == n).any()]
fig, ax = plt.subplots(figsize=(1.8 * len(names_with_flags) + 3, 4.5))
positions, data_boxes, colours = [], [], []
for i, name in enumerate(names_with_flags):
    for j, outcome in enumerate(['True positive', 'False positive']):
        selected = flags[(flags['dataset'] == name) & (flags['outcome'] == outcome)]
        values = selected['region_enrichment'].to_numpy()
        if len(values):
            positions.append(3 * i + j)
            data_boxes.append(values)
            colours.append(COLOURS[name] if outcome == 'True positive' else 'lightgrey')
parts = ax.boxplot(data_boxes, positions=positions, widths=0.8, patch_artist=True, showfliers=False)
for patch, colour in zip(parts['boxes'], colours):
    patch.set_facecolor(colour)
ax.axhline(1.0, color='black', linestyle=':', linewidth=1)
ax.set_xticks([3 * i + 0.5 for i in range(len(names_with_flags))])
ax.set_xticklabels(names_with_flags)
ax.set_ylabel(f'Anatomical enrichment of {MODEL_NAME} Grad-CAM\n(Share inside thorax divided by thorax area)')
ax.set_title('Coloured bars are true positives, grey bars are false positives')
plt.tight_layout()
plt.savefig(f'figures/{PREFIX}anatomical_enrichment.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
nih_patients = np.unique(datasets['NIH']['patients'])
calibration_patients = set(np.random.default_rng(SEED).permutation(nih_patients)[:len(nih_patients) // 2])

def aurc(confidence, correct):
    order = np.argsort(-confidence, kind='stable')
    risk = np.cumsum(1 - correct[order]) / np.arange(1, len(order) + 1)
    return float(risk.mean())

def risk_coverage(confidence, correct, points=200):
    order = np.argsort(-confidence, kind='stable')
    risk = np.cumsum(1 - correct[order]) / np.arange(1, len(order) + 1)
    keep = np.unique(np.linspace(0, len(order) - 1, min(points, len(order))).astype(int))
    return (keep + 1) / len(order), risk[keep]

def delta(first, second):
    return None if first is None or second is None else first - second

def aurc_at(rows, values, correct):
    return aurc(values[rows], correct[rows])

def delta_auc_at(rows, correct, full, base):
    return delta(safe_auc(correct[rows], full[rows]), safe_auc(correct[rows], base[rows]))

def delta_aurc_at(rows, correct, full, base):
    return aurc(full[rows], correct[rows]) - aurc(base[rows], correct[rows])

In [ ]:
failure_rows, curves, coefficients = [], {}, []
for scope, decision_value in (('Flagged findings', 1), ('Unflagged findings', 0)):
    frame = decisions[(decisions['decision'] == decision_value) & has_enrichment].copy()
    frame['abs_margin'] = frame['margin'].abs()
    nih_part = np.where(frame['patient'].isin(calibration_patients), 'NIH calibration', 'NIH evaluation')
    frame['partition'] = np.where(frame['dataset'] != 'NIH', frame['dataset'], nih_part)    
    calibration = frame[frame['partition'] == 'NIH calibration']
    frame['Confidence margin'] = frame['abs_margin']
    frame['TTA agreement'] = -frame['tta_gap']
    frame['Anatomical enrichment'] = frame['region_enrichment']
    for label, features in COMBINERS:
        scaler = StandardScaler().fit(calibration[features])
        combiner = LogisticRegression(max_iter=1000).fit(scaler.transform(calibration[features]), calibration['correct'])
        frame[label] = combiner.predict_proba(scaler.transform(frame[features]))[:, 1]
        weights = {feature: float(weight) for feature, weight in zip(features, combiner.coef_[0])}
        coefficients.append({'scope': scope, 'model': label, **weights})
    for part in [p for p in pd.unique(frame['partition']) if p != 'NIH calibration']:
        subset = frame[frame['partition'] == part]
        correct = subset['correct'].to_numpy()
        failures = int((1 - correct).sum())
        if failures < MIN_EVENTS or correct.sum() < MIN_EVENTS:
            print(f"{scope} on {part}: {failures} failures among {len(subset)}. Too few to evaluate.")
            continue
        scores = {label: subset[label].to_numpy(dtype=float) for label in SCORES}
        statistics = {}
        for label, score in scores.items():
            statistics[(label, 'AUROC_f')] = (auc_against, {'correct': correct, 'values': score})
            statistics[(label, 'AURC')] = (aurc_at, {'values': score, 'correct': correct})
        base, full = scores['Margin + TTA'], scores['Margin + TTA + anatomy']
        added = {'correct': correct, 'full': full, 'base': base}
        statistics[('Anatomy added', 'Delta AUROC_f')] = (delta_auc_at, added)
        statistics[('Anatomy added', 'Delta AURC')] = (delta_aurc_at, added)
        ci = bootstrap(statistics, patient_groups(subset['patient']))
        everything = np.arange(len(subset))
        for key, (statistic, arguments) in statistics.items():
            failure_rows.append({
                'scope': scope,
                'partition': part,
                'score': key[0],
                'metric': key[1],
                'value': statistic(everything, **arguments),
                'low': ci[key][0],
                'high': ci[key][1],
                'n': len(subset),
                'failures': failures,
            })
        curves[(scope, part)] = {label: risk_coverage(score, correct) for label, score in scores.items()}

In [ ]:
failure = pd.DataFrame(failure_rows)
failure.to_csv(f'models/{PREFIX}xai_failure_detection.csv', index=False)
pd.DataFrame(coefficients).to_csv(f'models/{PREFIX}xai_combiner_coefficients.csv', index=False)
print("Standardised logistic coefficients:")
print(pd.DataFrame(coefficients).round(3).to_string(index=False))
for (scope, part), _ in curves.items():
    table = failure[(failure['scope'] == scope) & (failure['partition'] == part)]
    print(f"\n{scope} on {part}: n={table['n'].iloc[0]}, failures={table['failures'].iloc[0]}")
    for row in table.itertuples():
        print(f"  {row.score:<28}{row.metric:<15}{row.value:.4f} [{row.low:.4f}, {row.high:.4f}]")

In [ ]:
panels = [key for key in curves if key[0] == 'Flagged findings']
if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4), squeeze=False)
    for ax, key in zip(axes[0], panels):
        for label, (coverage, risk) in curves[key].items():
            ax.plot(coverage, risk, label=label, linewidth=1.5)
        ax.set_title(f"{key[1]}: {MODEL_NAME} flagged findings")
        ax.set_xlabel('Coverage (share of flags kept)')
        ax.set_ylabel('False-flag rate among kept flags')
        ax.grid(alpha=0.3)
    axes[0][0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f'figures/{PREFIX}risk_coverage.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
nih_count = len(datasets['NIH']['labels'])
sample_rows = np.sort(np.random.default_rng(SEED).choice(nih_count, min(RANDOMISATION_IMAGES, nih_count), replace=False))
sample_images = np.asarray(datasets['NIH']['images'][sample_rows], dtype=np.float32)
sample_targets = np.argmax(nih_probabilities[sample_rows], axis=1)

def target_maps(model, images, targets):
    model_forward = make_forward(model)
    maps = []
    for start in range(0, len(images), BATCH_SIZE):
        cams, _ = model_forward(images[start:start + BATCH_SIZE])
        chosen = targets[start:start + BATCH_SIZE]
        maps.append(cams[np.arange(len(chosen)), :, :, chosen])
    return np.concatenate(maps)

def randomise(layer, seed):
    layer.kernel.assign(tf.keras.initializers.GlorotUniform(seed=seed)(layer.kernel.shape))

def rank_similarity(first, second):
    pairs = [(a, b) for a, b in zip(first, second) if np.ptp(a) > 0 and np.ptp(b) > 0]
    values = [spearmanr(a.ravel(), b.ravel()).statistic for a, b in pairs]
    return np.asarray(values, dtype=float)

In [ ]:
trained_maps = target_maps(classifier, sample_images, sample_targets)
randomised_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
stages = {'Trained model (reference)': trained_maps}
randomise(randomised_model.get_layer('logits'), SEED)
stages['Classifier randomised'] = target_maps(randomised_model, sample_images, sample_targets)
if MODEL == 'vit':
    block = randomised_model.get_layer('vit_b16_base').get_layer('vit_encoder').encoder_layers[-1]
    attention, mlp = block.mha, block.mlp
    top_layers = dict(enumerate([attention.query_dense, attention.key_dense, attention.value_dense, attention.output_dense, mlp.dense_1, mlp.dense_2]))
    top_stage = '+ Final transformer block randomised'
else:
    base_layers = enumerate(randomised_model.get_layer('densenet121_base').layers)
    top_layers = {i: layer for i, layer in base_layers if isinstance(layer, tf.keras.layers.Conv2D) and layer.name.startswith('conv5')}
    top_stage = '+ Dense block 4 randomised'
for i, layer in top_layers.items():
    randomise(layer, SEED + i)
stages[top_stage] = target_maps(randomised_model, sample_images, sample_targets)
randomisation_rows = []
for stage, maps in stages.items():
    similarity = rank_similarity(trained_maps, maps)
    has_values = len(similarity) > 0
    low = float(np.percentile(similarity, 2.5)) if has_values else np.nan
    high = float(np.percentile(similarity, 97.5)) if has_values else np.nan
    randomisation_rows.append({
        'stage': stage,
        'mean_spearman': float(similarity.mean()) if has_values else np.nan,
        'low': low,
        'high': high,
        'n': len(similarity),
    })
randomisation = pd.DataFrame(randomisation_rows)
randomisation.to_csv(f'models/{PREFIX}xai_randomisation.csv', index=False)
print("Model-randomisation sanity check:")
print(randomisation.round(3).to_string(index=False))

In [ ]:
means = randomisation['mean_spearman'].to_numpy(dtype=float)
low = randomisation['low'].to_numpy(dtype=float)
high = randomisation['high'].to_numpy(dtype=float)
errors = np.nan_to_num(np.vstack([means - low, high - means]))
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.bar(randomisation['stage'], means, yerr=errors, capsize=4, color=['#7F7F7F'] + ['#4C72B0'] * (len(means) - 1))
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylim(min(-0.2, float(np.nanmin(randomisation['low'])) - 0.05), 1.05)
ax.set_ylabel('Spearman correlation with trained Grad-CAM')
subtitle = 'Bars show the mean, whiskers the 2.5th to 97.5th percentile across images'
ax.set_title(f'{MODEL_NAME} Grad-CAM under cascading weight randomisation\n({subtitle})', fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(f'figures/{PREFIX}randomisation_check.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
background_rows = np.sort(np.random.default_rng(SEED + 1).choice(nih_count, min(SHAP_BACKGROUND, nih_count), replace=False))
background = np.asarray(datasets['NIH']['images'][background_rows], dtype=np.float32)
shap_explainers = {}

def predict_rgb(batch):
    return classifier.predict(np.ascontiguousarray(batch[..., :1], dtype=np.float32), batch_size=BATCH_SIZE, verbose=0)

def select_class(z, c):
    return z[:, c:c + 1]

def shap_explainer(c):
    if c not in shap_explainers:
        selector = tf.keras.layers.Lambda(select_class, arguments={'c': c}, name=f'select_{c}')
        head = tf.keras.Model([classifier.input], selector(classifier.output))
        shap_explainers[c] = shap.GradientExplainer(head, background, batch_size=BATCH_SIZE)
    return shap_explainers[c]

def explain_image(image, classes):
    rgb = np.repeat(image.astype(np.uint8), 3, axis=-1)
    explainer = lime_image.LimeImageExplainer(random_state=SEED)
    explanation = explainer.explain_instance(rgb, predict_rgb, labels=classes, **LIME_SETTINGS)
    maps = {}
    for c in classes:
        weights = np.zeros(explanation.segments.max() + 1, dtype=np.float32)
        for segment, weight in explanation.local_exp.get(c, []):
            weights[segment] = weight
        values = shap_explainer(c).shap_values(image[None].astype(np.float32), nsamples=SHAP_SAMPLES, rseed=SEED)
        shap_map = np.asarray(values, dtype=np.float32).reshape(IMG_SIZE, IMG_SIZE)
        maps[c] = {'LIME': weights[explanation.segments], 'SHAP': shap_map}
    return maps

In [ ]:
def enrichment(attribution, layers):
    region = (layers & 1).astype(bool)
    positive = np.maximum(attribution, 0.0)
    if region.sum() == 0:
        return np.nan
    if positive.sum() <= 0:
        return 1.0
    return float(positive[region].sum() / positive.sum() / region.mean())

In [ ]:
cross = []
for name, data in datasets.items():
    if data['anatomy'] is None:
        continue
    pool = flags[flags['dataset'] == name]
    if name == 'NIH':
        pool = pool[~pool['patient'].isin(calibration_patients)]
    candidates = pd.unique(pool['row'])
    if len(candidates) == 0:
        continue
    take = min(CROSS_METHOD_IMAGES, len(candidates))
    chosen = np.sort(np.random.default_rng(SEED).choice(candidates, take, replace=False))
    print(f"{name}: running LIME and SHAP on {len(chosen)} of {len(candidates)} images with flagged findings.")
    for count, row in enumerate(chosen):
        image = np.asarray(data['images'][row], dtype=np.float32)
        layers = np.asarray(data['anatomy'][row])
        image_flags = pool[pool['row'] == row]
        maps = explain_image(image, sorted({int(c) for c in image_flags['driver']}))
        for flag in image_flags.itertuples():
            driver_maps = maps[int(flag.driver)]
            cross.append({
                'dataset': name,
                'row': int(row),
                'patient': flag.patient,
                'finding': flag.finding,
                'driver': int(flag.driver),
                'correct': int(flag.correct),
                'Grad-CAM': float(flag.region_enrichment),
                'LIME': enrichment(driver_maps['LIME'], layers),
                'SHAP': enrichment(driver_maps['SHAP'], layers),
            })
        if count % 50 == 0:
            print(f"  Explained {count} of {len(chosen)} images", flush=True)
cross = pd.DataFrame(cross)
cross.to_csv(f'models/{PREFIX}xai_cross_method.csv', index=False)

In [ ]:
cross_rows = []
for name in pd.unique(cross['dataset']):
    part = cross[cross['dataset'] == name].dropna(subset=['Grad-CAM', 'LIME', 'SHAP'])
    agreement = part[['Grad-CAM', 'LIME', 'SHAP']].corr(method='spearman')
    print(f"\n{name}: {len(part)} flagged findings, {int((1 - part['correct']).sum())} false.")
    print(f"  Spearman Grad-CAM to LIME {agreement.at['Grad-CAM', 'LIME']:.3f}")
    print(f"  Spearman Grad-CAM to SHAP {agreement.at['Grad-CAM', 'SHAP']:.3f}")
    print(f"  Spearman LIME to SHAP {agreement.at['LIME', 'SHAP']:.3f}")    
    correct = part['correct'].to_numpy()
    statistics = {} 
    for method in ('Grad-CAM', 'LIME', 'SHAP'):
        values = part[method].to_numpy(dtype=float)
        statistics[method] = (auc_against, {'correct': correct, 'values': values})
    ci = bootstrap(statistics, patient_groups(part['patient']))
    everything = np.arange(len(part))
    for method, (statistic, arguments) in statistics.items():
        value = statistic(everything, **arguments)
        cross_rows.append({
            'dataset': name,
            'method': method,
            'auroc_f': value,
            'low': ci[method][0],
            'high': ci[method][1],
            'rho_gradcam': float(agreement.at['Grad-CAM', method]),
            'n': len(part),
        })
        auroc = value if value is not None else float('nan')
        print(f"  {method}: AUROC_f {auroc:.3f} [{ci[method][0]:.3f}, {ci[method][1]:.3f}]")
cross_summary = pd.DataFrame(cross_rows)
cross_summary.to_csv(f'models/{PREFIX}xai_cross_method_summary.csv', index=False)

In [ ]:
def outline(ax, mask, colour):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    for contour in contours:
        closed = np.vstack([contour[:, 0, :], contour[:1, 0, :]])
        ax.plot(closed[:, 0], closed[:, 1], color=colour, linewidth=1)

def heat(ax, image, attribution, title):
    ax.imshow(image, cmap='gray')
    positive = np.maximum(attribution, 0.0)
    if positive.max() > 0:
        ax.imshow(positive / positive.max(), cmap='jet', alpha=0.45)
    ax.set_title(title, fontsize=8)
    ax.axis('off')

In [ ]:
examples = []
for name in pd.unique(cross['dataset']):
    part = cross[cross['dataset'] == name]
    for outcome in (1, 0):
        group = part[part['correct'] == outcome]
        if len(group):
            distance = (group['Grad-CAM'] - group['Grad-CAM'].median()).abs()
            examples.append(group.iloc[int(distance.to_numpy().argmin())])
if examples:
    fig, axes = plt.subplots(len(examples), 4, figsize=(13, 3.4 * len(examples)), squeeze=False)
    for axis_row, example in zip(axes, examples):
        data = datasets[example['dataset']]
        row, driver = int(example['row']), int(example['driver'])
        image = np.asarray(data['images'][row], dtype=np.float32)
        layers = np.asarray(data['anatomy'][row])
        maps = explain_image(image, [driver])[driver]
        outcome = 'true positive' if example['correct'] == 1 else 'false positive'
        axis_row[0].imshow(image[:, :, 0], cmap='gray')
        outline(axis_row[0], layers & 1, 'cyan')
        truth = ground_truth.get(example['dataset'], {}).get((row, example['finding']))
        if truth is not None:
            outline(axis_row[0], truth, 'yellow')
        axis_row[0].set_title(f"{example['dataset']}: {example['finding']} ({outcome})", fontsize=8)
        axis_row[0].axis('off')
        grey = image[:, :, 0]
        cam = upsample(outputs[example['dataset']]['cams'][row, :, :, driver])
        heat(axis_row[1], grey, cam, f"Grad-CAM, enrichment {example['Grad-CAM']:.2f}")
        heat(axis_row[2], grey, maps['LIME'], f"LIME, enrichment {enrichment(maps['LIME'], layers):.2f}")
        heat(axis_row[3], grey, maps['SHAP'], f"SHAP, enrichment {enrichment(maps['SHAP'], layers):.2f}")
    caption = 'Thoracic region in cyan, radiologist annotation in yellow'
    plt.suptitle(f'Typical {MODEL_NAME} true and false positives. {caption}', y=1.0, fontsize=10)
    plt.tight_layout()
    plt.savefig(f'figures/{PREFIX}qualitative_examples.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
results = {
    'classification_transfer': transfer.to_dict('records'),
    'localisation': localisation_summary.to_dict('records'),
    'enrichment_shift': shift.to_dict('records'),
    'failure_detection': failure.to_dict('records'),
    'combiner_coefficients': coefficients,
    'randomisation': randomisation.to_dict('records'),
    'cross_method': cross_summary.to_dict('records'),
}
with open(f'models/{PREFIX}xai_results.json', 'w') as handle:
    json.dump(results, handle, indent=2, default=float)
print("Saved every table to models/ and every figure to figures/.")